In [23]:
import os
import pandas as pd
from tqdm import tqdm

In [67]:
# Path to the root folder containing subdirectories
input_root_folder = "/data/weichen/st_datasets/world_st_traffic/processed_data_outlier"
output_root_folder = "/data/weichen/st_datasets/world_st_traffic/pretrain_datasets"

detectors = pd.read_csv("detectors_public.csv")

# Initialize variables to store the total number of rows (T) and details
total_rows = 0
file_shapes = []

def process_column_name(col):
    try:
        # 尝试将列名转换为浮点数，再转换为整数，然后转换为字符串
        col_int = int(float(col))
        # 如果转换后的整数长度为 7，则在前面加上 '0'
        if len(str(col_int)) == 7:
            return '0' + str(col_int)
        else:
            return str(col_int)  # 返回原始列名或转换后的列名
    except ValueError:
        return col  # 如果转换失败，返回原始列名

# Traverse through the root folder to process CSV files
for subdir, _, files in tqdm(os.walk(input_root_folder), desc="Processing files"):
    for file_name in files:
        if file_name.endswith('.csv'):
            
            dataset_name = file_name.split('.')[0]
            city = dataset_name.split('_')[0]
            type = dataset_name.split('_')[1]
            
            if city != "losangeles" and city != "stuttgart":
                continue
            
            if city == "losangeles":
                city = "losanageles"
            
            # 读取 CSV 文件
            file_path = os.path.join(subdir, file_name)
            df = pd.read_csv(file_path)
            
            # 先将 time 列转换为 datetime 类型
            df['time'] = pd.to_datetime(df['time'])
            
            # 将 time 列设置为索引
            df.set_index('time', inplace=True)

            # 使用 resample 将数据采样到5分钟的频率，这里取均值 （下采样）
            df_5min = df.resample('5min').mean()
            
            # 将缺失值进行线性插值 （上采样） 
            df_5min = df_5min.interpolate(method='linear')
            
            if not os.path.exists(f"{output_root_folder}/{dataset_name}"):
                os.makedirs(f"{output_root_folder}/{dataset_name}")
            
            if city == "stuttgart":
                df_5min.columns = df_5min.columns.map(process_column_name)
            
            # 保存文件
            df_5min.to_pickle(f'{output_root_folder}/{dataset_name}/{dataset_name}_temporal.pkl')
            
            detector_ids = list(df_5min.columns)
            
            detectors_city = detectors[detectors["citycode"] == city].copy()

            selected_detectors = detectors_city[detectors_city["detid"].isin(detector_ids)].copy()

            selected_detectors = (selected_detectors.drop_duplicates(subset="detid").set_index("detid").reindex(detector_ids).reset_index())[["detid", "lat", "long"]].copy()

            selected_detectors.rename(columns={"detid": "ID", "lat": "Latitude", "long": "Longitude"}, inplace=True)

            selected_detectors.to_pickle(f'{output_root_folder}/{dataset_name}/{dataset_name}_spatial.pkl')
            

Processing files: 40it [00:01, 23.31it/s]


### new process

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import seaborn as sns

# Path to the root folder containing subdirectories
input_root_folder = "/data/weichen/st_datasets/world_st_traffic/processed_data_outlier"
output_root_folder = "/data/weichen/st_datasets/world_st_traffic/pretrain_datasets"

detectors = pd.read_csv("detectors_public.csv")

def process_column_name(col):
    try:
        col_int = int(float(col))
        if len(str(col_int)) == 7:
            return '0' + str(col_int)
        else:
            return str(col_int)
    except ValueError:
        return col

# Function to split data into valid segments based on missing time gaps
def split_segments(df, min_gap):
    gaps = (df.index[1:] - df.index[:-1]).total_seconds() / 60  # Calculate gaps in minutes
    break_points = [-1] + (gaps > min_gap).nonzero()[0].tolist() + [len(df)-1]
    segments = [df.iloc[break_points[i]+1:break_points[i + 1]+1] for i in range(len(break_points) - 1)]
    return [seg for seg in segments if not seg.empty]

# Minimum gap to consider as a missing time block
min_gap_minutes = 60  # Adjust based on your specific dataset


# Function to plot mean and standard deviation with discontinuities
def plot_with_discontinuities(means, std_devs, output_path, title):
    plt.figure(figsize=(20, 6))
    segments = split_segments(pd.DataFrame({'mean': means, 'std': std_devs}), min_gap_minutes)
    
    for segment in segments:
        mean_segment = segment['mean']
        std_segment = segment['std']
        plt.plot(mean_segment.index, mean_segment, label='Mean', color='red')
        plt.fill_between(mean_segment.index, mean_segment - std_segment, mean_segment + std_segment, color='orange', alpha=0.5, label='Standard Deviation')
    
    plt.title(title)
    plt.xlabel('Date')
    plt.ylabel('Values')
    # plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()


# Traverse through the root folder to process CSV files
for subdir, _, files in tqdm(os.walk(input_root_folder), desc="Processing files"):
    for file_name in files:
        if file_name.endswith('.csv'):
            dataset_name = file_name.split('.')[0]
            city = dataset_name.split('_')[0]
            type = dataset_name.split('_')[1]
            
            if city == "losangeles":
                city = "losanageles"

            file_path = os.path.join(subdir, file_name)
            df = pd.read_csv(file_path)

            # Convert 'time' column to datetime and set as index
            df['time'] = pd.to_datetime(df['time'])
            df.set_index('time', inplace=True)

            # Determine original sampling rate
            original_sampling_rate = (df.index[1] - df.index[0]).total_seconds() / 60

            # Split data into valid segments
            segments = split_segments(df, min_gap_minutes)

            # Process each segment
            processed_segments = []
            for segment in segments:                
                resampled_segment = segment.resample('5min').mean()
                
                # First fill NaN at the beginning and end using forward and backward fill
                resampled_segment = resampled_segment.ffill().bfill()
                
                interpolated_segment = resampled_segment.interpolate(method='linear')
                processed_segments.append(interpolated_segment)

            # Concatenate processed segments
            df_processed = pd.concat(processed_segments)
            
            if type == 'occ':  # Multiply flow values by 10 if type is 'occ'
                df_processed *= 100
            
            output_dir = f"{output_root_folder}/{dataset_name}"
            if not os.path.exists(output_dir):
                os.makedirs(output_dir)

            if city == "stuttgart":
                df_processed.columns = df_processed.columns.map(process_column_name)

            # Save temporal data
            df_processed.to_pickle(f'{output_dir}/{dataset_name}_temporal.pkl')
            
            # Count NaNs in df_processed
            total_nans = df_processed.isna().sum().sum()

            # Log or print the NaN statistics
            print(f'Total NaNs in df_processed: {total_nans} in {city} {type}')
            

            detector_ids = list(df_processed.columns)
            detectors_city = detectors[detectors["citycode"] == city].copy()
            selected_detectors = detectors_city[detectors_city["detid"].isin(detector_ids)].copy()
            selected_detectors = (
                selected_detectors.drop_duplicates(subset="detid")
                .set_index("detid")
                .reindex(detector_ids)
                .reset_index()
            )[["detid", "lat", "long"]].copy()
            selected_detectors.rename(columns={"detid": "ID", "lat": "Latitude", "long": "Longitude"}, inplace=True)

            # Save spatial data
            selected_detectors.to_pickle(f'{output_dir}/{dataset_name}_spatial.pkl')
            
            
            # 绘制热力图
            value_min = df_processed.min().min()
            value_max = df_processed.max().max()

            plt.figure(figsize=(12, 6))
            sns.heatmap(df_processed.T, cbar=True, cmap="viridis", xticklabels=False, yticklabels=False, vmin=value_min, vmax=value_max)
            plt.title(f"Heatmap for {city}")
            plt.savefig(f'{output_dir}/{dataset_name}_heatmap.png')
            plt.close()
            

            # 计算每个时间点的平均值和标准差
            means = df_processed.mean(axis=1)
            std_devs = df_processed.std(axis=1)

            # # 绘制平均值和标准差图
            # plt.figure(figsize=(20, 6))
            # plt.plot(means.index, means, label='Mean', color='red')
            # plt.fill_between(means.index, means - std_devs, means + std_devs, color='orange', alpha=0.5, label='Standard Deviation')
            # plt.title(f'{city}')
            # plt.xlabel('Date')
            # plt.ylabel('Values')
            # plt.legend()
            # plt.grid(True)

            # # 保存平均值图
            # plt.savefig(f'{output_dir}/{dataset_name}_vis.png')
            # plt.close()
            
            # 绘制平均值和标准差图
            plot_with_discontinuities(means, std_devs, f'{output_dir}/{dataset_name}_vis.png', title=f'{city}')

Processing files: 0it [00:00, ?it/s]

Total NaNs in df_processed: 0 in utrecht flow


Processing files: 2it [00:01,  1.40it/s]

Total NaNs in df_processed: 0 in bolton occ
Total NaNs in df_processed: 0 in bolton speed
Total NaNs in df_processed: 0 in bolton flow


Processing files: 3it [00:02,  1.00it/s]

...

Processing files: 38it [03:34,  9.80s/it]

Total NaNs in df_processed: 0 in manchester occ
Total NaNs in df_processed: 0 in manchester speed
Total NaNs in df_processed: 0 in manchester flow


Processing files: 39it [03:38,  7.93s/it]

Total NaNs in df_processed: 0 in bordeaux occ
Total NaNs in df_processed: 0 in bordeaux flow


Processing files: 40it [03:40,  5.51s/it]
Processing files: 40it [03:40,  5.51s/it]
